In [1]:
import numpy as np
import matplotlib.pyplot as plt
import orekit
from orekit.pyhelpers import setup_orekit_curdir
from org.orekit.utils import Constants

# ==========================================
# 1. OREKIT ENVIRONMENT INITIALIZATION
# ==========================================
orekit.initVM()
setup_orekit_curdir('C:/Users/Simone/Downloads/orekit-data-main')

# ==========================================
# 2. CONSTANTS AND INPUT DATA
# ==========================================
# Earth Physical Constants (from Orekit, converted to km)
mu_E = Constants.WGS84_EARTH_MU / 1e9             # Earth's gravitational parameter [km^3/s^2]
R_E = Constants.WGS84_EARTH_EQUATORIAL_RADIUS / 1000.0  # Earth's mean equatorial radius [km]

# Moon Physical Constants (Explicit for Patched Conics)
mu_M = 4902.8000                # Moon's gravitational parameter [km^3/s^2]
R_M = 1737.4                    # Moon's mean radius [km]
D_EM = 384400.0                 # Mean Earth-Moon distance [km]

# Initial Orbit Parameters (LEO - Low Earth Orbit)
h_LEO = 300.0                   # Earth parking altitude [km]
r1 = R_E + h_LEO                # Initial orbit radius [km]
v_LEO = np.sqrt(mu_E / r1)      # LEO circular velocity [km/s]

# Final Orbit Parameters (LLO - Low Lunar Orbit)
h_LLO = 100.0                   # Lunar parking altitude [km]
r2 = R_M + h_LLO                # Target lunar orbit radius [km]
v_LLO = np.sqrt(mu_M / r2)      # LLO circular velocity [km/s]

print("========================================================")
print("  1. PARKING ORBITS PARAMETERS")
print("========================================================")
print(f"LEO Velocity (Earth): {v_LEO:.4f} km/s")
print(f"LLO Velocity (Moon):  {v_LLO:.4f} km/s\n")

# ==========================================
# 3. PHASE 1: TRANS-LUNAR INJECTION (TLI)
# ==========================================
r_apogeo_tx = D_EM
a_tx = (r1 + r_apogeo_tx) / 2.0  # Semi-major axis of the transfer ellipse

# Velocity calculation on the ellipse (Vis-viva equation)
v_tx_perigeo = np.sqrt(mu_E * (2/r1 - 1/a_tx))
v_tx_apogeo = np.sqrt(mu_E * (2/r_apogeo_tx - 1/a_tx))

# Departure maneuver Delta-V
DeltaV_TLI = v_tx_perigeo - v_LEO

# Time of flight (One-way, hence half orbital period)
TOF_sec = np.pi * np.sqrt((a_tx**3) / mu_E)
TOF_days = TOF_sec / 86400.0

print("========================================================")
print("  2. EARTH DEPARTURE (TRANS-LUNAR INJECTION)")
print("========================================================")
print(f"Required perigee velocity: {v_tx_perigeo:.4f} km/s")
print(f"TLI Delta-V (Burn 1):      {DeltaV_TLI:.4f} km/s")
print(f"Time of Flight (TOF):      {TOF_days:.2f} days\n")

# ==========================================
# 4. PHASE 2: MOON ARRIVAL AND LUNAR ORBIT INSERTION (LOI)
# ==========================================
v_Moon = np.sqrt(mu_E / D_EM)     # Moon's orbital velocity [km/s]
v_inf = v_Moon - v_tx_apogeo      # Hyperbolic excess (V_infinity) [km/s]

# Energy conservation for the lunar hyperbola
v_iperbole_perilenio = np.sqrt(v_inf**2 + 2*mu_M / r2)

# Delta-V for capture
DeltaV_LOI = v_iperbole_perilenio - v_LLO

print("========================================================")
print("  3. MOON ARRIVAL (LUNAR ORBIT INSERTION)")
print("========================================================")
print(f"Moon Velocity:                   {v_Moon:.4f} km/s")
print(f"Hyperbolic Excess (V_inf):       {v_inf:.4f} km/s")
print(f"Hyperbola velocity at perilune:  {v_iperbole_perilenio:.4f} km/s")
print(f"LOI Delta-V (Burn 2 - Braking):  {DeltaV_LOI:.4f} km/s\n")

# ==========================================
# 5. FINAL DELTA-V BUDGET RESULTS
# ==========================================
DeltaV_Tot = DeltaV_TLI + DeltaV_LOI

print("========================================================")
print("  TOTAL DELTA-V BUDGET ")
print("========================================================")
print(f"Departure Delta-V (TLI): {DeltaV_TLI:.4f} km/s")
print(f"Capture Delta-V (LOI):   {DeltaV_LOI:.4f} km/s")
print(f"TOTAL DELTA-V:           {DeltaV_Tot:.4f} km/s")
print("========================================================")

  1. PARKING ORBITS PARAMETERS
LEO Velocity (Earth): 7.7258 km/s
LLO Velocity (Moon):  1.6335 km/s

  2. EARTH DEPARTURE (TRANS-LUNAR INJECTION)
Required perigee velocity: 10.8322 km/s
TLI Delta-V (Burn 1):      3.1064 km/s
Time of Flight (TOF):      4.98 days

  3. MOON ARRIVAL (LUNAR ORBIT INSERTION)
Moon Velocity:                   1.0183 km/s
Hyperbolic Excess (V_inf):       0.8301 km/s
Hyperbola velocity at perilune:  2.4547 km/s
LOI Delta-V (Burn 2 - Braking):  0.8212 km/s

  TOTAL DELTA-V BUDGET 
Departure Delta-V (TLI): 3.1064 km/s
Capture Delta-V (LOI):   0.8212 km/s
TOTAL DELTA-V:           3.9277 km/s
